# Clase 2 — Automatic Speech Recognition (ASR) con Whisper y síntesis de voz (TTS)

## Pregunta central

> **¿Cómo convertimos voz en texto y texto en voz?**

## Idea principal

El reconocimiento automático de voz (ASR) transforma audio en texto; la síntesis de voz (TTS) hace el camino inverso. Juntos forman la base de un asistente conversacional: el usuario habla, el sistema entiende, responde y "habla" de vuelta.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Explicar el pipeline de ASR (audio → features → modelo → texto).
- Transcribir un audio con Whisper usando `transformers`.
- Sintetizar voz en español con un modelo TTS local.
- Armar un pipeline integrado voz → texto → respuesta → voz.
- Reconocer los límites y riesgos de privacidad del audio en salud.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Qué es ASR y cómo funciona |
| 2 | Transcribir con Whisper |
| 3 | Qué es TTS y cómo funciona |
| 4 | Sintetizar voz en español |
| 5 | Pipeline integrado: asistente de consultorio |
| 6 | Límites y privacidad |
| 7 | Actividad grupal: recepción inteligente de mensajes |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. La primera ejecución descarga los modelos (Whisper Tiny y MMS-TTS). Después quedan en caché.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** este par ASR + TTS es el esqueleto de los agentes conversacionales que agendan turnos y gestionan recetas en el Track Salud.


## Glosario mínimo

| Término | Explicación breve |
|---|---|
| ASR | Reconocimiento automático de voz: audio → texto |
| TTS | Síntesis de voz: texto → audio |
| Whisper | Modelo de ASR de OpenAI, disponible en varios tamaños |
| Whisper Tiny | Versión más chica de Whisper (~39M parámetros) |
| Pipeline | Función de `transformers` que une modelo + pre/postprocesamiento |
| Transcripción | Texto que resulta de convertir audio en palabras |
| Alucinación | Texto inventado por el modelo cuando no está seguro |
| Sample rate | Muestras por segundo; Whisper espera 16 kHz |
| MMS-TTS | Modelo de síntesis de voz multilingüe de Meta |
| Latencia | Tiempo entre entrada y salida |
| Consentimiento | Permiso explícito para usar datos (clave en salud) |


---
## 1. Qué es ASR y cómo funciona

El **reconocimiento automático de voz (ASR)** convierte una señal de audio en texto.

> **Analogía del transcriptor.** Un ASR es como un transcriptor que escucha una grabación y escribe lo que escucha. Pero en vez de oídos, usa un modelo entrenado con muchísimas horas de audio y texto.

El pipeline tiene varias etapas:

```text
audio (waveform)
    |
    v
features (espectrograma / mel)
    |
    v
modelo (red neuronal)
    |
    v
texto transcrito
```

> **Pensalo así:** en la clase 1 vimos cómo convertir audio en espectrograma. Whisper hace eso internamente y luego usa una red neuronal para "leer" ese espectrograma y producir texto.

### Errores típicos del ASR

| Error | Qué pasa | Ejemplo |
|---|---|---|
| Homófonos | Palabras que suenan igual | "haya" vs "halla" |
| Ruido de fondo | El modelo se confunde | Música de espera transcrita |
| Alucinación | Inventa texto | Silencio → frase inventada |
| Nombres propios | Difíciles de transcribir | Apellidos poco comunes |

> **Importante:** la transcripción no es perfecta. En salud, un error puede cambiar el sentido de un dato. Por eso siempre hay que validar antes de usar el texto para tomar decisiones.


---
## 2. Transcribir con Whisper

Vamos a usar **Whisper Tiny**, la versión más chica de Whisper. Es lo suficientemente liviana para correr en CPU y alcanza para experimentar.

> **Pensalo así:** Whisper viene en tamaños (tiny, base, small, medium, large). Más grande = más preciso pero más lento y pesado. Para una clase en CPU, tiny es el equilibrio correcto.

La primera ejecución descarga el modelo (~75 MB). Después queda en caché.


In [ ]:
# --- Cargar el pipeline de ASR ---
import os
import numpy as np
import soundfile as sf
from transformers import pipeline
#import warnings
#warnings.filterwarnings("ignore")

# Ruta del audio de ejemplo.
RUTA_AUDIO = os.path.join("..", "assets", "audio", "taxi.wav")

# Creamos el pipeline de reconocimiento de voz con Whisper Tiny.
# device=0 usaría GPU; dejamos CPU por defecto.
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=-1,
)

print("Pipeline ASR listo (Whisper Tiny).")


### Transcribir el audio

El pipeline acepta la ruta del archivo o el arreglo de muestras. Le pasamos la ruta y el sample rate correcto (16 kHz).


In [ ]:
# --- Transcribir el audio de ejemplo ---
from IPython.display import Audio, display

print("Audio de entrada:")
display(Audio(filename=RUTA_AUDIO))

resultado_taxi = asr(RUTA_AUDIO, return_timestamps = True)

print("Transcripción de Whisper:")
print("-" * 50)
print(resultado_taxi["text"])
print("-" * 50)


---
## 3. Qué es TTS y cómo funciona

La **síntesis de voz (TTS)** convierte texto en audio hablado.

> **Analogía del locutor.** Un TTS es como un locutor que lee un guion. Pero en vez de una persona, es un modelo que genera la forma de onda del habla a partir del texto.

```text
texto
    |
    v
modelo TTS
    |
    v
audio (waveform)
    |
    v
archivo WAV
```

> **Pensalo así:** el ASR "lee" audio y escribe texto; el TTS "lee" texto y escribe audio. Son espejos.

### Usos en salud

| Uso | Ejemplo |
|---|---|
| Confirmar turnos | "Su turno es el jueves a las 10" |
| Leer recetas | "Tome una pastilla cada 8 horas" |
| Accesibilidad | Leer indicaciones a pacientes con dificultad visual |
| Recordatorios | Llamadas automáticas de recordatorio |


---
## 4. Sintetizar voz en español

Vamos a usar **MMS-TTS** de Meta, un modelo multilingüe que incluye español. Es chico (~100 MB) y corre en CPU.

> **Pensalo así:** MMS-TTS es como un locutor multilingüe. Le decís el idioma (español) y el texto, y genera el audio.


In [ ]:
# --- Cargar el pipeline de TTS ---
tts = pipeline(
    "text-to-speech",
    model="facebook/mms-tts-spa",
    device=-1,
)

print("Pipeline TTS listo (MMS-TTS español).")


### Sintetizar una frase

El pipeline TTS devuelve un objeto con el audio (arreglo numpy) y el sample rate.


In [ ]:
# --- Sintetizar una frase de ejemplo ---
frase = "Su turno es el jueves a las diez de la mañana."

salida_tts = tts(frase)

# El resultado tiene el audio y el sample rate.
audio_tts = salida_tts["audio"]
sr_tts = salida_tts["sampling_rate"]

print(f"Audio generado: {len(audio_tts):,} muestras a {sr_tts} Hz")
print(f"Duración: {len(audio_tts) / sr_tts:.2f} s")

# --- Guardar el audio sintetizado como WAV ---
import os

# Carpeta de salida para los audios generados.
os.makedirs("salidas", exist_ok=True)
ruta_tts = os.path.join("salidas", "respuesta_tts.wav")
sf.write(ruta_tts, audio_tts, sr_tts)

print("Audio guardado en:", ruta_tts)

### Escuchar y verificar

> **Pregunta de interpretación:** ¿La voz suena natural? ¿Se entiende la frase? ¿Qué limitaciones notás (entonación, velocidad, pausas)?

> **Pensalo así:** el TTS genera audio que podés guardar y reproducir. En una app, ese audio se reproduce al usuario. La calidad depende del modelo y del texto.


In [ ]:
# --- Visualizar la waveform del audio sintetizado ---
import matplotlib.pyplot as plt

tiempo_tts = np.arange(len(audio_tts)) / sr_tts
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(tiempo_tts, audio_tts, color="tab:green", lw=0.5)
ax.set_title("Waveform del audio sintetizado (TTS)")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Amplitud")
ax.set_xlim(0, len(audio_tts) / sr_tts)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Audio sintetizado:")
display(Audio(data=audio_tts, rate=sr_tts))


---
## 5. Pipeline integrado: asistente de consultorio

Ahora unimos todo: una persona hace una consulta de turnos, el sistema la transcribe (ASR), genera una respuesta (texto) y la "lee" (TTS).

Para que el ejemplo sea reproducible, vamos a escribir una consulta normal, convertirla en audio con TTS y usar ese audio como entrada del pipeline. El audio taxi.wav queda separado como demostración de ASR con ruido.

```text
consulta de turnos en audio
    |
    v
ASR (Whisper) -> texto
    |
    v
lógica de la app -> respuesta
    |
    v
TTS (MMS) -> audio de respuesta
    |
    v
reproducción al usuario
```

> **Pensalo así:** este es el esqueleto de un asistente conversacional. En la actividad siguiente reemplazamos la regla simple por un LLM pequeño que recibe la transcripción y propone una respuesta controlada.


In [ ]:
# --- Crear una consulta de turnos y convertirla en audio ---
# Esta consulta es la entrada del pipeline del consultorio.
MENSAJE_TURNO = "Hola, quisiera pedir un turno con la dermatóloga para el viernes por la mañana."
audio_consulta = tts(MENSAJE_TURNO)
ruta_consulta_turno = os.path.join("salidas", "consulta_turno.wav")
sf.write(
    ruta_consulta_turno,
    audio_consulta["audio"],
    audio_consulta["sampling_rate"],
)

print("Texto original:", MENSAJE_TURNO) 
print("Consulta de turnos guardada en:", ruta_consulta_turno)
display(Audio(filename=ruta_consulta_turno))

# --- Función que arma el pipeline completo ---
def asistente_voz(ruta_audio, respuesta_texto):
    """Transcribe un audio, arma una respuesta y la convierte en voz."""
    # 1) ASR: audio -> texto
    transcripcion = asr(ruta_audio, return_timestamps=True)["text"]

    # 2) Lógica simple: en la actividad la reemplazaremos por un LLM.
    salida = tts(respuesta_texto)

    return {
        "transcripcion": transcripcion,
        "respuesta": respuesta_texto,
        "audio": salida["audio"],
        "sr": salida["sampling_rate"],
    }

# La respuesta es informativa: no confirma un turno que la aplicación
# todavía no consultó ni registró en un sistema real.
respuesta_turno = "Recibimos su solicitud de turno. La recepción verificará la disponibilidad y le responderá."
resultado_asistente = asistente_voz(ruta_consulta_turno, respuesta_turno)

print("Transcripción de la consulta:", resultado_asistente["transcripcion"])
print("Respuesta del asistente:     ", resultado_asistente["respuesta"])
print(f"Audio de respuesta: {len(resultado_asistente['audio']):,} muestras")
display(Audio(data=resultado_asistente["audio"], rate=resultado_asistente["sr"]))


In [ ]:
# --- Guardar la respuesta del asistente ---
ruta_respuesta = os.path.join("salidas", "respuesta_asistente.wav")
sf.write(
    ruta_respuesta,
    resultado_asistente["audio"],
    resultado_asistente["sr"],
)
print("Respuesta del asistente guardada en:", ruta_respuesta)


---
## 6. Límites y privacidad

En salud, el audio es un dato sensible. Antes de construir un asistente, hay que considerar:

| Riesgo | Qué implica | Mitigación |
|---|---|---|
| Errores de transcripción | Un dato mal transcrito cambia el sentido | Validación humana en decisiones críticas |
| Datos clínicos sensibles | El audio puede contener información de salud | Consentimiento explícito |
| Almacenamiento | Guardar audio es guardar datos personales | Anonimizar, retener lo mínimo |
| Alucinaciones | El modelo inventa texto | No usar la transcripción como verdad absoluta |
| Latencia | El usuario espera respuesta | Optimizar modelos o usar streaming |

> **Importante:** un asistente de consultorio puede **agendar turnos** (bajo riesgo) pero no debería **diagnosticar** sin supervisión. El nivel de riesgo define cuánta automatización es aceptable.

> **Pensalo así:** el ASR + TTS es la "voz" del sistema. La "decisión" (qué responder) es otra capa, y ahí es donde más cuidado hay que tener en salud.


---
## 7. Actividad grupal — recepción inteligente de mensajes

### Desafío

Un consultorio recibe mensajes de voz por WhatsApp. Diseñen un flujo que use **Whisper para transcribir** y un **LLM pequeño para interpretar el mensaje y redactar una primera respuesta**.

El asistente no agenda turnos ni da diagnósticos: organiza la información para que la aplicación pueda decidir qué hacer después.

> El audio taxi.wav no representa un mensaje de consultorio: úselo sólo como ejemplo de una transcripción ruidosa. La entrada principal de esta actividad es la consulta de turnos que generamos en la sección 5 y que Whisper vuelve a transcribir. Los casos de prueba que siguen simulan otros mensajes reales y permiten distinguir un problema de Whisper de un problema del prompt o del LLM.

### Organización y tiempo: 30 minutos

Trabajen en grupos de 3 o 4 personas:

- **5 min — Contrato de salida:** definan qué información necesita recibir la aplicación.
- **10 min — Prompt:** escriban y mejoren el prompt del sistema para el asistente.
- **10 min — Pruebas:** prueben mensajes completos, incompletos, médicos y fuera de tema. Cambien el prompt cuando encuentren un problema.
- **5 min — Puesta en común:** compartan un caso que funcionó y otro que obligó a ajustar el diseño.

### Contrato mínimo

La respuesta del LLM debe indicar:

- **INTENCION**: turno, receta, horario, consulta_medica u otro.
- **DATOS_FALTANTES**: qué información debería pedir la aplicación.
- **RESPUESTA**: un mensaje breve, amable y claro para la persona.
- **ESCALAR**: si cuando el caso requiere intervención humana, especialmente ante una consulta médica o una respuesta dudosa.

### Preguntas para orientar el diseño

- ¿Qué datos necesita la recepción para continuar con cada intención?
- ¿Qué cosas el asistente no debe inventar ni prometer?
- ¿Cuándo conviene pedir una aclaración y cuándo escalar a una persona?
- ¿Qué ocurre si Whisper transcribe mal el mensaje?
- ¿Qué parte del resultado usaría la aplicación: la respuesta completa o campos separados?

> **Entrega:** un prompt de sistema, una decisión sobre el formato de salida y una breve demostración con dos casos. Si llegan a tiempo, conviertan la respuesta final en audio usando el bloque de TTS de la clase.


In [ ]:
# ✏️ PASO 1: Cargá el LLM pequeño que usará el asistente.
# La primera ejecución descarga el modelo y puede tardar unos minutos.
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

REPO_ID_LLM = "unsloth/LFM2.5-1.2B-Instruct-GGUF"
FILENAME_LLM = "LFM2.5-1.2B-Instruct-Q4_0.gguf"

ruta_modelo_llm = hf_hub_download(
    repo_id=REPO_ID_LLM,
    filename=FILENAME_LLM,
)

llm = Llama(
    model_path=ruta_modelo_llm,
    n_ctx=2048,
    n_gpu_layers=0,
    verbose=False,
)

def preguntar_llm(mensaje, system_prompt, temperature=0.2, max_tokens=180):
    respuesta = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": mensaje},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return respuesta["choices"][0]["message"]["content"].strip()

print("LLM listo para probar.")


In [ ]:
# ✏️ PASO 2: Escribí el prompt y probalo con la transcripción de Whisper.
# El grupo puede modificar este prompt y observar cómo cambia la salida.
SYSTEM_PROMPT_EQUIPO = """
Sos el asistente de recepción de un consultorio.
Ordená cada mensaje para que otra parte de la aplicación pueda decidir qué hacer.
No diagnostiques, no indiques tratamientos, no inventes horarios ni confirmes acciones que no ejecutaste.
Si falta información, indicá qué dato hay que pedir. Si hay una consulta médica,
una urgencia o no entendés el mensaje, marcá ESCALAR como si.

Respondé exactamente con estas cuatro líneas y nada más:
INTENCION: turno | receta | horario | consulta_medica | otro
DATOS_FALTANTES: lista breve o ninguno
RESPUESTA: mensaje breve, amable y claro para la persona
ESCALAR: si | no
"""

transcripcion_whisper = resultado_asistente["transcripcion"]
print("Texto original de la consulta:")
print(MENSAJE_TURNO)
print("\nTranscripción de la consulta obtenida por Whisper:")
print(transcripcion_whisper)

respuesta_sobre_audio = preguntar_llm(
    transcripcion_whisper,
    system_prompt=SYSTEM_PROMPT_EQUIPO,
)

print("\nInterpretación del LLM sobre la consulta de turnos:")
print(respuesta_sobre_audio)


In [ ]:
# ✏️ PASO 3: Probá el prompt con distintos casos y registrá los resultados.
# Busquen errores: respuestas inventadas, datos faltantes ignorados o casos
# que deberían escalarse a una persona.
CASOS_PRUEBA = {
    "turno con datos": "Necesito un turno con la dermatóloga para el martes por la tarde.",
    "turno incompleto": "¿Me pueden dar un turno, por favor?",
    "consulta médica": "Tengo dolor fuerte desde ayer, ¿qué medicamento debería tomar?",
    "fuera de tema": "Quería saber si el consultorio tiene estacionamiento.",
}

respuestas_prueba = {}
for nombre_caso, mensaje in CASOS_PRUEBA.items():
    respuesta = preguntar_llm(
        mensaje,
        system_prompt=SYSTEM_PROMPT_EQUIPO,
    )
    respuestas_prueba[nombre_caso] = respuesta
    print(f"\n--- {nombre_caso} ---")
    print("Mensaje:", mensaje)
    print(respuesta)

print("\nAhora elijan un caso y, si llegan a tiempo, conviertan solo el campo RESPUESTA en audio con TTS.")


### Tabla de reflexión

| Pregunta | Respuesta |
|---|---|
| ¿Qué formato de salida le conviene a la aplicación y por qué? | |
| ¿Qué cambio en el prompt mejoró el resultado? | |
| ¿En qué caso el asistente pidió datos faltantes? | |
| ¿En qué caso debería escalar a una persona? | |
| ¿Qué problema puede producir una mala transcripción de Whisper? | |
| ¿Qué riesgo de privacidad existe al guardar el audio o la transcripción? | |

> **Cierre:** Whisper convierte voz en texto y el LLM transforma ese texto en una decisión estructurada y una respuesta. La aplicación puede usar esos campos para continuar el flujo, registrar el mensaje o derivarlo a una persona.


---

## Síntesis de la clase

- El ASR convierte audio en texto; el TTS convierte texto en audio.
- Whisper Tiny transcribe en español y corre en CPU.
- MMS-TTS sintetiza voz en español de forma local.
- El pipeline voz → texto → respuesta → voz es el esqueleto de un asistente.
- Un LLM pequeño puede interpretar la transcripción y devolver campos que la aplicación use para continuar el flujo.
- El prompt y el formato de salida también forman parte del diseño de una solución de software.
- En salud, los errores de transcripción y la privacidad del audio son críticos.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

La clase 3 abre el NLP clínico: tokenización, embeddings de texto y extracción de entidades. Ahí veremos cómo el texto que produce el ASR se convierte en datos estructurados (ej: "el paciente tiene diabetes" → entidad clínica).

## Conexión con el track

Salud usará ASR + TTS para asistentes conversacionales, y el NLP de la clase 3 para entender el texto transcrito.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.
